# Parcel label OCR on Colab (WeVisOne-VL + PP-OCRv6)

1. **Runtime → Change runtime type → GPU**. The free T4 works with `float16`; L4 or A100 (Colab Pro) can use `bfloat16`.
2. Run the cells from top to bottom. Installation takes about 5 minutes, and the first model download (about 9 GB) takes a few more.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Get the code
If the repo is private, use a GitHub token (`https://<token>@github.com/...`), or upload a zip of the repo and unzip it.

In [ ]:
REPO_URL = "https://github.com/jacobkastrup/vision_tests.git"  # @param {type:"string"}
BRANCH = "claude/adoring-dirac-2bbjtm"  # @param {type:"string"}
!git clone -b $BRANCH $REPO_URL /content/vision_tests
%cd /content/vision_tests
!./setup.sh

After `setup.sh` finishes, use **Runtime → Restart session** once, because it replaces Colab's preinstalled torch. Then continue below.

In [ ]:
%cd /content/vision_tests
import torch
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print(torch.__version__, torch.cuda.get_device_name(0), DTYPE)

## Upload label images

In [ ]:
import os
from google.colab import files
os.makedirs("labels", exist_ok=True)
for name, data in files.upload().items():
    open(f"labels/{name}", "wb").write(data)
!ls labels

## Read one field from one label: compare all modes and both engines

In [ ]:
IMAGE = "labels/" + sorted(os.listdir("labels"))[0]
!python read_label.py "$IMAGE" "the tracking number under the barcode" "the recipient address block" \
    --mode all --engine both --dtype $DTYPE --debug outputs/debug.png

In [ ]:
from IPython.display import Image as Show
Show("outputs/debug.png", width=600)

## Batch with ground truth
Edit `fields.example.json` and optionally upload an `expected.csv` (columns: `image` plus one column per field).

In [ ]:
!python batch_read.py labels fields.example.json -o outputs/results.csv \
    --mode crop,filter,full --engine both --dtype $DTYPE --debug-dir outputs/debug \
    $( [ -f expected.csv ] && echo --expected expected.csv )

In [ ]:
import pandas as pd
pd.read_csv("outputs/results.csv")

In [ ]:
!zip -qr outputs.zip outputs && echo ok
files.download("outputs.zip")